# 06 — Kernel Memory in-process : la couche d'abstraction Microsoft

Jusqu'ici cette série a manipulé l'infrastructure **à la main** : des embeddings calculés
exprès ([03](03-Embeddings-From-Scratch.ipynb)), des upserts et des recherches Qdrant
écrits directement ([01](01-Hands-On-Grounding.ipynb), [05](05-Stockage-Vectoriel.ipynb)),
du retrieval mesuré sur un gold français ([02](02-Retrieval-Avance.ipynb)). C'est le
meilleur moyen de comprendre *ce que fait* un backend de mémoire sémantique — mais ce
n'est pas ce qu'on écrit en production : on délègue le pipeline à une couche d'abstraction.

**Kernel Memory** (Microsoft, [github.com/microsoft/kernel-memory](https://github.com/microsoft/kernel-memory))
est cette couche : ingestion de documents (texte, Markdown, PDF, Word...), découpage en
partitions, embeddings, indexation vectorielle, et recherche qui renvoie des **citations**
— fichier source, partition, passage, score de pertinence. C'est la pièce qui manquait
entre notre Qdrant brut et les agents Semantic Kernel qui consomment la mémoire.

Dans ce notebook nous utilisons le mode **in-process** (`MemoryServerless`) : le pipeline
tourne dans le processus du notebook, sans service web ni conteneur, avec :

| Brique | Choix | Pourquoi |
|---|---|---|
| Embeddings | `granite-embedding-107m-multilingual` (IBM) en GGUF Q8_0 via LLamaSharp CPU | modèle **local** multilingue conçu pour le RAG, cohérent avec la convention modèles-locaux de la série (cf. 02 : `sentence_transformers`) |
| Stockage fichiers | `SimpleFileStorage` (dossier temporaire) | zéro dépendance, démontrable |
| Index vectoriel | `SimpleVectorDb` (recherche exacte brute) | zéro conteneur — le compromis ANN est traité dans [05](05-Stockage-Vectoriel.ipynb) |

**Plan** : (1) pipeline et modèle local, (2) ingestion d'un corpus français avec tags,
(3) sous le capot — le partitionnement `TextChunker`, (4) recherche avec citations,
(5) mesure — la granularité des partitions contre le rappel, (6) limites et exercices.

## 1. Le pipeline Kernel Memory

Une ingestion KM enchaîne des **étapes** (`extract` → `partition` → `embed` → `index`)
sur chaque document : extraction du texte, découpage en partitions, vectorisation,
écriture dans le store vectoriel. Une recherche vectorise la question et renvoie les
partitions les plus proches, **avec leur provenance** — c'est la capacité distinctive
de KM par rapport à un upsert Qdrant écrit à la main : la traçabilité
document → partition → passage est portée par le pipeline, pas reconstruite après coup.

Côté packages : `Microsoft.KernelMemory.Core` (le pipeline), l'intégration
`LLamaSharp.kernel-memory` (l'inférence locale llama.cpp pour les embeddings) et le
backend CPU de LLamaSharp. Notons un fait d'écosystème utile : en 0.98, le connecteur
ONNX de KM ne couvre que la **génération** de texte — pour des embeddings 100 % locaux,
le chemin officiel de l'écosystème est LLamaSharp.

In [1]:
#r "nuget: Microsoft.KernelMemory.Core, 0.98.250508.3"
#r "nuget: LLamaSharp.kernel-memory, 0.27.0"
#r "nuget: LLamaSharp.Backend.Cpu, 0.27.0"

using System;
using System.IO;
using System.Linq;
using System.Net.Http;
using System.Collections.Generic;
using System.Threading.Tasks;
using System.Text.RegularExpressions;
using Microsoft.KernelMemory;
using Microsoft.KernelMemory.Configuration;

Console.WriteLine("Microsoft.KernelMemory.Core 0.98.250508.3 + LLamaSharp 0.27.0 (CPU) charges (NuGet)");

Installing Packages LLamaSharp.Backend.Cpu LLamaSharp.kernel-memory Microsoft.KernelMemory.Core

Microsoft.KernelMemory.Core 0.98.250508.3 + LLamaSharp 0.27.0 (CPU) charges (NuGet)


## 2. Le modèle d'embeddings local (GGUF via llama.cpp)

Nous téléchargeons **`granite-embedding-107m-multilingual`** (IBM, 2025) — un embedder
compact multilingue **conçu pour le RAG** — au format GGUF quantisé Q8_0 (121 Mo), depuis
la conversion de référence de `bartowski`. L'inférence passe par llama.cpp en CPU via
LLamaSharp : aucune clé d'API, aucun service externe, cohérent avec la convention
modèles-locaux de la série. Le téléchargement est **mis en cache** dans un dossier
temporaire : une ré-exécution le saute.

Remarque honnête : les embedders « à instructions » recommandent souvent de préfixer
requêtes et passages différemment. Nous utilisons le modèle en mode **symétrique**
(texte brut des deux côtés) — fonctionnel — et l'**exercice 3** demande d'implémenter
les préfixes recommandés et de mesurer ce qu'ils apportent réellement sur notre gold.

In [2]:
// Telechargement en cache (dossier temporaire, non commite) -- chemins affiches en basename.
string modelDir = Path.Combine(Path.GetTempPath(), "km_rag06_gguf");
Directory.CreateDirectory(modelDir);
string ggufPath = Path.Combine(modelDir, "granite-embedding-107m-multilingual-Q8_0.gguf");

async Task DownloadIfAbsentAsync(string url, string dest)
{
    if (File.Exists(dest) && new FileInfo(dest).Length > 0)
    {
        Console.WriteLine($"cache OK   : {Path.GetFileName(dest)} ({new FileInfo(dest).Length / (1024 * 1024)} Mo)");
        return;
    }
    using var http = new HttpClient();
    using var resp = await http.GetAsync(url, HttpCompletionOption.ResponseHeadersRead);
    resp.EnsureSuccessStatusCode();
    await using var src = await resp.Content.ReadAsStreamAsync();
    await using var dst = File.Create(dest + ".part");
    var buffer = new byte[1 << 20];
    long read = 0; int n;
    while ((n = await src.ReadAsync(buffer)) > 0)
    {
        await dst.WriteAsync(buffer.AsMemory(0, n));
        read += n;
    }
    await dst.DisposeAsync();
    File.Move(dest + ".part", dest, overwrite: true);
    Console.WriteLine($"telecharge : {Path.GetFileName(dest)} ({read / (1024 * 1024)} Mo)");
}

const string GGUF_URL = "https://huggingface.co/bartowski/granite-embedding-107m-multilingual-GGUF/resolve/main/granite-embedding-107m-multilingual-Q8_0.gguf";
await DownloadIfAbsentAsync(GGUF_URL, ggufPath);
Console.WriteLine($"modele pret dans le cache : {new DirectoryInfo(modelDir).Name}/");

cache OK   : granite-embedding-107m-multilingual-Q8_0.gguf (115 Mo)


modele pret dans le cache : km_rag06_gguf/


In [3]:
// En mode notebook, le chargeur standard de LLamaSharp ne sonde pas le dossier natif du
// package backend : on pointe explicitement le DLL via NativeLibraryConfig (doit etre
// configure AVANT le premier appel a l'API native), avec repli noavx si AVX2 est absent.
string backendPkg = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages", "llamasharp.backend.cpu", "0.27.0");
string nativeDir = Path.Combine(backendPkg, "LLamaSharpRuntimes", "win-x64", "native", "avx2");
if (!File.Exists(Path.Combine(nativeDir, "llama.dll")))
    nativeDir = Path.Combine(backendPkg, "LLamaSharpRuntimes", "win-x64", "native", "noavx");
LLama.Native.NativeLibraryConfig.All.WithLibrary(Path.Combine(nativeDir, "llama.dll"), Path.Combine(nativeDir, "mtmd.dll"));
Console.WriteLine($"backend natif LLamaSharp : {new DirectoryInfo(nativeDir).Parent!.Name}/{new DirectoryInfo(nativeDir).Name}");

backend natif LLamaSharp : native/avx2


In [4]:
// Construction du pipeline in-process : LLamaSharp fournit le generateur d'embeddings.
using LLamaSharp.KernelMemory;
using Microsoft.KernelMemory.AI;
using Microsoft.Extensions.DependencyInjection;
using System.Runtime.CompilerServices;
using System.Threading;

// KM 0.98 : l'orchestrateur in-process exige un ITextGenerator resolu par injection de
// dependances, meme quand aucune generation de texte n'est demandee (pipeline 100 %
// embeddings). Ce generateur minimal n'est jamais invoque dans ce notebook.
#pragma warning disable KMEXP00 // ITextTokenizer est marque experimental dans KM 0.98
class GenerateurTexteInactif : ITextGenerator
{
    public int MaxTokenTotal => 512;

    public int CountTokens(string text) => text.Length / 4; // heuristique ~4 caracteres/jeton

    public IReadOnlyList<string> GetTokens(string text) => new[] { text };

    public async IAsyncEnumerable<GeneratedTextContent> GenerateTextAsync(
        string prompt, TextGenerationOptions options,
        [EnumeratorCancellation] CancellationToken ct)
    {
        await Task.CompletedTask;
        yield return new GeneratedTextContent("(generation de texte non configuree)", new TokenUsage());
    }
}

var llamaConfig = new LLamaSharpConfig(ggufPath) // modelPath en constructeur positionnel
{
    ContextSize = 2048, // CPU pur -- aucun GPU requis
    MainGpu = -1 // backend CPU : llama.cpp attend -1 (aucun device)
};

KernelMemoryBuilder NewBuilder()
{
    var b = new KernelMemoryBuilder();
    b.Services.AddSingleton<ITextGenerator>(new GenerateurTexteInactif());
    return b;
}

var memory = NewBuilder()
    .WithSimpleFileStorage(Path.Combine(Path.GetTempPath(), "km_rag06_store"))
    .WithSimpleVectorDb(Path.Combine(Path.GetTempPath(), "km_rag06_vecdb"))
    .WithLLamaSharpTextEmbeddingGeneration(llamaConfig)
    .Build<MemoryServerless>();

Console.WriteLine("Pipeline MemoryServerless pret : SimpleFileStorage + SimpleVectorDb + LLamaSharp (granite-embedding-107m, CPU)");

Pipeline MemoryServerless pret : SimpleFileStorage + SimpleVectorDb + LLamaSharp (granite-embedding-107m, CPU)



warning CS1701: En supposant que la référence d'assembly 'Microsoft.Extensions.DependencyInjection.Abstractions, Version=9.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' utilisée par 'Microsoft.KernelMemory.Core' correspond à l'identité 'Microsoft.Extensions.DependencyInjection.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' de 'Microsoft.Extensions.DependencyInjection.Abstractions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'Microsoft.Extensions.DependencyInjection.Abstractions, Version=9.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' utilisée par 'Microsoft.KernelMemory.Abstractions' correspond à l'identité 'Microsoft.Extensions.DependencyInjection.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' de 'Microsoft.Extensions.DependencyInjection.Abstractions', il se peut que vous deviez fournir une stratégie runtime



## 3. Corpus de test et ingestion

Huit documents français inspirés des thèmes réels de cette série (incidents, HNSW,
tokenisation, retrieval) — assez longs pour que le partitionnement soit discriminant.
Chaque fichier est écrit sur disque puis **ingéré via la pipeline ETL de KM** (c'est son
rôle : lire des *fichiers*, pas seulement des chaînes en mémoire), avec des **tags**
`theme` exploitables en filtrage (exercice 1).

In [5]:
var corpus = new (string file, string tags, string body)[]
{
    ("incident-perte-donnees.md", "incident,qdrant", string.Join("\n\n", new[]
    {
        "# Incident : la perte de donnees de mars",
        "En mars, la collection conversations de Qdrant a perdu environ 40 pourcents de ses points apres un redemarrage du conteneur. La cause racine : le volume Docker etait monte en ecriture par deux instances simultanees, et aucune sauvegarde n'avait ete restauree depuis onze jours. Le montant de perte reel a ete mesure par comparaison avec l'export nocturne du disque de persistance.",
        "Trois lecons en sont sorties. D'abord, le split-brain du montage doit ete detecte par une sonde qui compare l'UUID du conteneur a l'owner declare. Ensuite, la sauvegarde doit etre restauree periodiquement dans un environnement epheliere : une sauvegarde jamais restauree n'est pas une sauvegarde. Enfin, le journal d'incident est devenu un document de premiere classe de la base : la memoire semantique indexe aussi les echecs.",
        "Apels remediation : verrou proprietaire sur le volume, sauvegarde bi-quotidienne avec restauration automatique le dimanche, et ajout d'un compteur de points publie qui alarme au-dela de 5 pourcents d'ecart avec l'attendu."
    })),
    ("split-brain.md", "incident,qdrant", string.Join("\n\n", new[]
    {
        "# Anti split-brain sur Qdrant",
        "Le split-brain survient quand deux processus croient chacun detenir l'unicite d'un volume de stockage. Dans notre cas, un conteneur zombie conservait un descripteur de fichier ouvert sur le repertoire de persistance pendant que le nouveau conteneur reconstruisait l'index HNSW. Les ecritures des deux processus se sont entrelacees dans les segments memtables, et la compaction a ensuite produit des segments incoherents.",
        "La defense comporte trois etages : un verrou disque (fichier lock avec PID), une sonde d'unicite qui interroge l'API cluster de Qdrant et compare les identifiants de replica, et un test de restoration hebdomadaire qui demarre une instance sur une copie de la sauvegarde et compte les points. Sans le troisieme etage, les deux premiers donnent une fausse confiance : on detecte le split-brain, mais pas la corruption silencieuse qu'il laisse."
    })),
    ("hnsw-parametrage.md", "qdrant,indexation", string.Join("\n\n", new[]
    {
        "# Parametrage HNSW en pratique",
        "HNSW construit un graphe multicouche ou chaque point relie a ses M voisins les plus proches ; la recherche descend les couches en greedy search guidee par ef_construction. Le compromis central : M eleve ameliore le rappel des requetes difficiles mais renforce le cout memoire de l'index et le temps de construction. Pour nos collections de conversation, M = 16 et ef_construction = 200 se sont averes le point doux.",
        "Le parametre runtime ef (au moment de la requete) reste le levier dominant du rappel : monter ef de 64 a 256 a fait passer le rappel au top-10 de 0,88 a 0,97 sur notre gold interne, au prix d'une latence triplee. La quantization scalaire (TurboQuant) comprime les vecteurs d'un facteur quatre avec une perte de rappel mesurable mais acceptable au-dela de 100k points, la ou l'index ne tient plus en RAM.",
        "En dessous de 50k points et en local, l'index exact reste la reference : HNSW n'apporte son gain de latence qu'une fois le brut force vectoriel devenu le goulot. C'est exactement le compromis que le notebook 05 de cette serie mesure a la main."
    })),
    ("bpe-compte.md", "tokenisation,cout", string.Join("\n\n", new[]
    {
        "# BPE : le token, unite de compte",
        "La tokenisation BPE fusionne iterativement les paires de caracteres les plus frequentes du corpus d'entrainement en vocabulaire. Consequence directe pour une memoire semantique : chaque fragment de texte ingere, chaque requete, chaque passage stocke est facture en tokens, pas en caracteres. Un mot francais rare comme reconstruisaient coute trois a quatre tokens la ou l'article defini en coute un.",
        "Le notebook 04 de cette serie deroule BPE a la main ; ici on en retient la consequence economique : le cout d'une memoire semantique se pilote en tokens. Partitionner un document en blocs de 256 tokens au lieu de 64 divise par quatre le nombre d'embeddings a calculer et donc le cout d'ingestion, mais grossit chaque partition, ce que la section 5 de ce notebook mesure cote rappel.",
        "Regle pratique retenue : estimer le compte de tokens d'un texte francais par un quart de sa longueur en caracteres (approximation large, suffisante pour dimensionner un pipeline)."
    })),
    ("hyde.md", "retrieval", string.Join("\n\n", new[]
    {
        "# HyDE : embedder une reponse hypothetique",
        "HyDE (Hypothetical Document Embeddings) reformule la question en document hypothetique avant la recherche vectorielle : au lieu d'embedder la question brute, on genere d'abord un court passage fictif qui ressemblerait a la reponse, et on cherche ses voisins. L'intuition : l'espace des embeddings rapproche deux documents, pas une question et un document.",
        "Le notebook 02 mesure HyDE sur le gold francais : gain net sur les questions courtes et lexicalement eloignees du corpus, gain nul voire negatif sur les questions deja riches en vocabulaire du domaine, au prix d'un appel de generation supplementaire par requete. HyDE est donc un levier conditionnel, pas un defaut.",
        "Kernel Memory n'implemente pas HyDE nativement : c'est une transformation de requete, qui vit naturellement cote orchestrateur (serie SemanticKernel) avant l'appel au backend memoire."
    })),
    ("grounding.md", "retrieval,agents", string.Join("\n\n", new[]
    {
        "# Grounding : ancrer les agents dans les faits",
        "Le grounding demande a un agent generateur de citer ses sources : chaque affirmation doit remonter a un passage retrievable, et le pipeline fournit les citations avec leur partition d'origine pour verifier. Une reponse non ancree est traitee comme une hypothese, pas comme un fait : le distinguo est le coeur du cahier des charges de cette serie.",
        "En pratique, la boucle grounding comporte trois temps : retrieval des passages pertinents, generation contrainte a ne s'appuyer que sur ces passages, puis verification humaine ou automatique que chaque phrase de la reponse a une citation. Le taux d'affirmations ancrees est la metrique qui a remplace le simple taux de reponses correctes dans nos tableaux de bord."
    })),
    ("quantization-turboquant.md", "qdrant,cout", string.Join("\n\n", new[]
    {
        "# Quantization TurboQuant",
        "La quantization vectorielle encode chaque dimension sur moins de bits : int8 au lieu de float32 divise la taille de l'index par quatre. TurboQuant, la variante de Qdrant, reconstruit les vecteurs quantizes avec une correction d'erreur qui limite la perte de rappel a environ un a deux points sur nos collections de test.",
        "Le vrai critere de decision n'est pas la place disque mais la RAM : un index qui tient en memoire sert ses requetes en millisecondes, un index echange sur disque voit sa latence exploser. La quantization est donc le levier qui retarde le moment ou il faut payer soit plus de RAM, soit une migration vers un backend distribue.",
        "Regle de decision retenue dans cette serie : quantizer au-dela de 100k points, mesurer le rappel avant et apres sur un gold fige, et documenter la perte acceptee. Jamais quantizer a l'aveugle."
    })),
    ("sauvegardes.md", "incident,qdrant", string.Join("\n\n", new[]
    {
        "# Sauvegardes : a moitie cablees n'est pas cable",
        "L'incident de mars a revele que la sauvegarde etait a moitie cablee : le script d'export tournait bien chaque nuit, mais personne n'avait verifie que les fichiers produits etaient complets et restaurables. Une sauvegarde non testee est un confort psychologique, pas une procedure.",
        "La remediation a trois volets : un test de restoration automatique hebdomadaire qui demonte un conteneur epheliere sur la copie, un chiffre de completude (nombre de points restaures contre attendu) publie dans le journal, et une alerte si le test ne s'execute pas : une sauvegarde silencieuse est une sauvegarde morte."
    }))
};

string corpusDir = Path.Combine(Path.GetTempPath(), "km_rag06_corpus");
Directory.CreateDirectory(corpusDir);
foreach (var (file, _, body) in corpus)
    await File.WriteAllTextAsync(Path.Combine(corpusDir, file), body);

var sw = System.Diagnostics.Stopwatch.StartNew();
foreach (var (file, tags, _) in corpus)
{
    var tc = new TagCollection();
    foreach (var t in tags.Split(',')) tc.Add("theme", t);
    await memory.ImportDocumentAsync(Path.Combine(corpusDir, file), index: "rag06", tags: tc);
}
sw.Stop();
Console.WriteLine($"{corpus.Length} documents ingeres (extract, partition, embed, index) en {sw.Elapsed.TotalSeconds:F1} s");
Console.WriteLine("corpus : " + string.Join(", ", corpus.Select(c => c.file)));

8 documents ingeres (extract, partition, embed, index) en 1,8 s


corpus : incident-perte-donnees.md, split-brain.md, hnsw-parametrage.md, bpe-compte.md, hyde.md, grounding.md, quantization-turboquant.md, sauvegardes.md


## 4. Sous le capot : les partitions générees par la pipeline

Avant la vectorisation, KM découpe chaque document en **partitions** (les unités
recherchées et citées). En 0.98, ce découpage n'est pas exposé comme classe publique :
le levier de configuration est `TextPartitioningOptions` (notamment
`MaxTokensPerParagraph`), et la façon honnête de voir le résultat est d'interroger la
pipeline elle-même. Une recherche large sur le vocabulaire du corpus fait remonter les
citations avec leurs partitions — on en déduit la granularité réelle par document.

In [6]:
// Inspection : une requete large fait remonter les partitions reellement generees.
int ApproxTokens(string s) => Math.Max(1, s.Length / 4);

var sonde = await memory.SearchAsync("qdrant incident sauvegarde indexation token retrieval embedding", index: "rag06", limit: 20);
foreach (var g in sonde.Results.GroupBy(c => c.SourceName).OrderBy(g => g.Key))
{
    var tailles = g.SelectMany(c => c.Partitions).Select(p => ApproxTokens(p.Text)).OrderBy(x => x).ToList();
    Console.WriteLine($"{g.Key,-32} {tailles.Count} partitions, tailles {tailles.First()}..{tailles.Last()} tokens (approx. 1 token = 4 caracteres)");
}

bpe-compte.md                    3 partitions, tailles 249..249 tokens (approx. 1 token = 4 caracteres)


hnsw-parametrage.md              3 partitions, tailles 274..274 tokens (approx. 1 token = 4 caracteres)


hyde.md                          3 partitions, tailles 225..225 tokens (approx. 1 token = 4 caracteres)


incident-perte-donnees.md        3 partitions, tailles 268..268 tokens (approx. 1 token = 4 caracteres)


quantization-turboquant.md       2 partitions, tailles 217..217 tokens (approx. 1 token = 4 caracteres)


sauvegardes.md                   3 partitions, tailles 162..162 tokens (approx. 1 token = 4 caracteres)


split-brain.md                   3 partitions, tailles 223..223 tokens (approx. 1 token = 4 caracteres)


## 5. Recherche avec citations

C'est la valeur ajoutée visible de KM : chaque résultat embarque sa **provenance** —
fichier source, numéro de partition, texte du passage, score de pertinence. Comparer à
[01](01-Hands-On-Grounding.ipynb), où la reconstruction de la provenance était un
exercice manuel.

In [7]:
foreach (var q in new[]
{
    "Comment le verrou anti split-brain detecte-t-il un conteneur zombie ?",
    "Quel parametre runtime fait le plus gagner en rappel, et a quel prix ?",
    "Pourquoi une sauvegarde jamais restauree n'est-elle pas une sauvegarde ?",
})
{
    var res = await memory.SearchAsync(q, index: "rag06", limit: 2);
    // KM peut retourner le meme document en plusieurs citations (une par partition
    // retenue) : on deduplique par fichier pour un affichage lisible.
    var citations = res.Results.GroupBy(c => c.SourceName).Select(g => g.First()).Take(2);
    Console.WriteLine();
    Console.WriteLine($"Q : {q}");
    foreach (var cit in citations)
    {
        var p = cit.Partitions.OrderByDescending(x => x.Relevance).First();
        var extrait = Regex.Replace(p.Text, @"\s+", " ").Trim();
        extrait = extrait.Length > 100 ? extrait[..100] + "..." : extrait;
        Console.WriteLine($"   -> {cit.SourceName} (partition {p.PartitionNumber}) rel={p.Relevance:F3} : {extrait}");
    }
}

Q : Comment le verrou anti split-brain detecte-t-il un conteneur zombie ?


   -> split-brain.md (partition 0) rel=0,814 : # Anti split-brain sur Qdrant Le split-brain survient quand deux processus croient chacun detenir l'...


Q : Quel parametre runtime fait le plus gagner en rappel, et a quel prix ?


   -> hnsw-parametrage.md (partition 0) rel=0,711 : # Parametrage HNSW en pratique HNSW construit un graphe multicouche ou chaque point relie a ses M vo...


Q : Pourquoi une sauvegarde jamais restauree n'est-elle pas une sauvegarde ?


   -> sauvegardes.md (partition 0) rel=0,749 : # Sauvegardes : a moitie cablees n'est pas cable L'incident de mars a revele que la sauvegarde etait...


## 6. Mesure : granularité des partitions contre rappel

Question de dimensionnement (cf. le document `bpe-compte` : le partitionnement pilote
le coût d'ingestion) : des partitions **fines** (64 tokens) coûtent plus d'embeddings
mais citent plus précisément ; des partitions **larges** (256 tokens) coûtent moins
mais diluent le signal. Nous mesurons trois choses sur deux pipelines identiques ne
différant que par `MaxTokensPerParagraph` : le **coût d'ingestion** (nombre de vecteurs
créés — `SimpleVectorDb` écrit un fichier par vecteur, on peut les compter), le
**rappel@1** et le **rappel@3** d'un mini-gold (6 questions, document attendu connu),
et la **taille moyenne des extraits cités**.

Verdict attendu : pas de claim « BEATS » — une mesure pédagogique sur un petit corpus,
dont la méthodologie (gold figé, recall@k) est celle du [02](02-Retrieval-Avance.ipynb).

In [8]:
MemoryServerless BuildMemory(int maxTokensPerParagraph)
{
    return NewBuilder()
        .WithSimpleFileStorage(Path.Combine(Path.GetTempPath(), $"km_rag06_mesure_store_{maxTokensPerParagraph}"))
        .WithSimpleVectorDb(Path.Combine(Path.GetTempPath(), $"km_rag06_mesure_vec_{maxTokensPerParagraph}"))
        .WithLLamaSharpTextEmbeddingGeneration(llamaConfig)
        .With(new TextPartitioningOptions { MaxTokensPerParagraph = maxTokensPerParagraph, OverlappingTokens = maxTokensPerParagraph / 4 })
        .Build<MemoryServerless>();
}

var gold = new (string q, string attendu)[]
{
    ("Comment le verrou anti split-brain detecte-t-il un conteneur zombie ?", "split-brain.md"),
    ("Quel parametre runtime fait le plus gagner en rappel, et a quel prix ?", "hnsw-parametrage.md"),
    ("Pourquoi une sauvegarde jamais restauree n'est-elle pas une sauvegarde ?", "sauvegardes.md"),
    ("Combien de points la collection a-t-elle perdus lors de l'incident de mars ?", "incident-perte-donnees.md"),
    ("Quand faut-il activer la quantization vectorielle ?", "quantization-turboquant.md"),
    ("En quoi HyDE aide-t-il les questions courtes ?", "hyde.md"),
};

Console.WriteLine($"{"strategie",-10} {"maxTok",7} {"vecteurs",9} {"rappel@1",9} {"rappel@3",9} {"extrait moyen",14}");
foreach (var (nom, maxTok) in new[] { ("fines", 64), ("larges", 256) })
{
    // Reprise d'une execution anterieure impossible : comptage exact des vecteurs
    // exige un magasin vide (SimpleVectorDb = un fichier par vecteur, cf. section 4).
    var vecDir = Path.Combine(Path.GetTempPath(), $"km_rag06_mesure_vec_{maxTok}");
    var storeDir = Path.Combine(Path.GetTempPath(), $"km_rag06_mesure_store_{maxTok}");
    if (Directory.Exists(vecDir)) Directory.Delete(vecDir, recursive: true);
    if (Directory.Exists(storeDir)) Directory.Delete(storeDir, recursive: true);

    var mem = BuildMemory(maxTok);
    foreach (var (file, tags, _) in corpus)
    {
        var tc = new TagCollection();
        foreach (var t in tags.Split(',')) tc.Add("theme", t);
        await mem.ImportDocumentAsync(Path.Combine(corpusDir, file), index: $"rag06-{nom}", tags: tc);
    }
    int nbVec = Directory.GetFiles(vecDir, "*", SearchOption.AllDirectories).Length;
    int hits1 = 0, hits3 = 0;
    long chars = 0;
    foreach (var (q, attendu) in gold)
    {
        var res = await mem.SearchAsync(q, index: $"rag06-{nom}", limit: 3);
        var files = res.Results.Select(c => c.SourceName).ToList();
        if (files.Contains(attendu)) hits3++;
        if (files.Count > 0 && files[0] == attendu) hits1++;
        var top = res.Results[0];
        var p = top.Partitions.OrderByDescending(x => x.Relevance).First();
        chars += Regex.Replace(p.Text, @"\s+", " ").Trim().Length;
    }
    Console.WriteLine($"{nom,-10} {maxTok,7} {nbVec,9} {100.0 * hits1 / gold.Length,8:F0}% {100.0 * hits3 / gold.Length,8:F0}% {chars / gold.Length,11} car.");
}

strategie   maxTok  vecteurs  rappel@1  rappel@3  extrait moyen


fines           64        60      100%      100%         160 car.


larges         256        10      100%      100%         833 car.


**Lecture.** Trois enseignements chiffrés. (1) Le **coût d'ingestion explose avec la
finesse** : 60 vecteurs à créer pour des partitions de 64 tokens contre 10 pour des
partitions de 256 — six fois plus d'embeddings sur un corpus de huit documents *courts* ;
le ratio monte sur des documents longs. (2) Le **rappel sature** : les deux stratégies
retrouvent le bon document dans le top-3 — la dilution du signal par les partitions
larges ne coûte pas cher sur ce petit corpus aux documents thématiquement étanches. Le
compromis devient réel sur des documents longs et multi-thèmes, où une partition large
mélange deux sujets et fait remonter le mauvais passage : c'est ce que le gold étendu de
l'**exercice 3** permet de tester. (3) La différence **visible dès ce corpus** est la
taille des extraits cités : ~5× plus courts en partitions fines (833 contre 160 caractères en moyenne). Quand la citation
alimente un LLM, la granularité pilote directement le coût du contexte et la précision
de la citation.

À retenir : **la granularité est un paramètre de conception, pas un détail** — elle pilote
le coût d'ingestion (nombre d'embeddings), la taille des citations, et au-delà d'un
certain degré de dilution, le rappel.

## 7. Limites et prolongements

- **Mode symétrique** : nous vectorisons requêtes et passages sans les instructions de
  préfixe que certains embedders recommandent — l'**exercice 3** mesure ce que les
  préfixes apportent réellement.
- **`SimpleVectorDb` = recherche exacte brute** : parfait pour démontrer, coûteux
  au-delà de ~10⁴ vecteurs. Le compromis rappel/ANN est le sujet du
  [05](05-Stockage-Vectoriel.ipynb) ; brancher KM sur un Qdrant serveur ne demande que
  `.WithQdrant(...)` (cf. [05b](05b-Stockage-Vectoriel-Serveur.ipynb)).
- **Pas d'`Ask` ici** : répondre en langage naturel exige un connecteur de génération
  (LLM) — hors scope backend mémoire, traité par la série SemanticKernel.
- **Hybride BM25 + dense** : Kernel Memory service le propose ; la mesure du gain sur
  notre gold français restera à faire.

## Exercices

**Exercice 1 — Filtrer la recherche par tag.** Les documents sont ingérés avec des tags
`theme` (ex. `incident`, `qdrant`, `retrieval`). Utilisez `MemoryFilter` pour restreindre
la question de l'incident de mars aux seuls documents taggés `incident`, et vérifiez que
`split-brain.md` ne remonte plus sur la question du verrou quand on filtre sur `qdrant`
seul.

# Indice : new MemoryFilter().ByTag("theme", "incident"), passe en argument filter de SearchAsync

**Exercice 2 — Ingestion d'un PDF.** KM décode nativement PDF/Word. Générez un PDF
d'une page contenant un des documents du corpus, ingérez-le dans un index `rag06-pdf`,
et vérifiez par une recherche que les citations pointent bien dans le PDF.

# Etape 1 : produire le fichier .pdf -- Etape 2 : ImportDocumentAsync -- Etape 3 : SearchAsync + citations

**Exercice 3 — Préfixes d'instruction et rappel.** Implémentez un `ITextEmbeddingGenerator`
qui wrappe le générateur LLamaSharp en préfixant passages et requêtes des instructions
recommandées par la fiche du modèle (indice : deux instances de pipeline, ou un drapeau
togglé entre les phases), ré-indexez le corpus, et mesurez le rappel@3 du gold de la
section 6 avant / après. Verdict attendu : quantifié, honnête — gain, nul ou négatif.

In [9]:
// Exercice 1 -- filtrage par tag (a completer)
// Objectif : la question de l'incident de mars ne doit citer que des documents theme=incident.
Console.WriteLine("Exercice a completer : filtrage par tag");
// var filtre = new MemoryFilter().ByTag("theme", "incident");
// var res = await memory.SearchAsync("Combien de points perdus lors de l'incident de mars ?", index: "rag06", filter: filtre, limit: 3);

Exercice a completer : filtrage par tag


In [10]:
// Exercice 2 -- ingestion d'un PDF (a completer)
// Objectif : citations provenant d'un vrai fichier PDF decode par la pipeline KM.
Console.WriteLine("Exercice a completer : ingestion PDF");
// string pdfPath = ...; // produire un PDF d'une page
// await memory.ImportDocumentAsync(pdfPath, index: "rag06-pdf");

Exercice a completer : ingestion PDF


In [11]:
// Exercice 3 -- prefixes d'instruction (a completer)
// Objectif : mesurer le rappel@3 du gold avec les instructions de prefixe respectees.
Console.WriteLine("Exercice a completer : prefixes d'instruction et rappel");
// class PrefixedGenerator : ITextEmbeddingGenerator { ... prefixe selon la phase ... }

Exercice a completer : prefixes d'instruction et rappel


## Conclusion

Kernel Memory in-process livre en un pipeline ce que les notebooks 01-05 ont construit
brique par brique : ETL documentaire, partitionnement, embeddings et **citations
traçables** — sans service ni conteneur, avec un embedder local GGUF sur CPU. La mesure
de la section 6 fixe le réflexe de conception : *la granularité des partitions est un
compromis coût/précision à décider explicitement*.

La suite logique de l'EPIC ([#13421](https://github.com/jsboige/CoursIA/issues/13421)) :
la recherche **hybride** BM25 + dense sur le gold français étendu, puis l'ingestion
multimodale — sur le même socle.